In [1]:
from the_well.data.normalization import ZScoreNormalization
from pathlib import Path
import numpy as np
import pickle
import fsspec
import torch
import yaml
import sys
import re

from pathlib import Path as _Path
PROJECT_ROOT = str(next(p for p in [_Path.cwd(), *_Path.cwd().parents] if (p / "pyproject.toml").exists()))  # repo root, works on laptop and cluster
sys.path.insert(0, PROJECT_ROOT)

from wavelet_flow_matching.utils.well_data import get_well_dataset, get_well_inputs
from wavelet_flow_matching.modules.wavelet_fm import WaveletFlowModule

print("Imports OK | MPS:", torch.backends.mps.is_available())
print(f"MPS allocated : {torch.mps.current_allocated_memory() / 1e9:.2f} GB")
print(f"MPS reserved  : {torch.mps.driver_allocated_memory()  / 1e9:.2f} GB")

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

Imports OK | MPS: True
MPS allocated : 0.00 GB
MPS reserved  : 0.00 GB


In [2]:
well_base_path    = f"{PROJECT_ROOT}/data/datasets"

#well_dataset_name = 'planetswe'
#well_dataset_name = 'rayleigh_benard'
well_dataset_name = 'turbulent_radiative_layer_2D'

stats_path        = f"{well_base_path}/{well_dataset_name}/stats.yaml"

# planetswe checkpoints
#ckpt_name = ''
#ckpt_name = ''

# rayleigh_benard checkpoints
#ckpt_name = f"WaveletFlow_{well_dataset_name}_3scales_db2_v1.0_20260401_150336-epoch=194-val_loss=0.0207_portable.ckpt"
#ckpt_name = f'WaveletFlow_{well_dataset_name}_1scale_db2_v1.0_20260406_075915-epoch=193-val_loss=0.0157_portable.ckpt'

# turbulent_radiative_layer_2D checkpoints
#ckpt_name = f'WaveletFlow_{well_dataset_name}_3scales_db2_v1.0_20260406_073804-epoch=193-val_loss=0.0190_portable.ckpt'
#ckpt_name = f'WaveletFlow_{well_dataset_name}_1scale_db2_v1.0_20260406_074619-epoch=189-val_loss=0.0094_portable.ckpt'
ckpt_name = f'WaveletFlow_{well_dataset_name}_data_space_v1.0_portable.ckpt'

ckpt_base_name    = re.split(r"_\d{8}_", ckpt_name)[0]
ckpt_path         = f"{PROJECT_ROOT}/experiments/{well_dataset_name}/ckpts/{ckpt_name}"

In [3]:
with fsspec.open(stats_path, mode='r') as f:
    stats = yaml.safe_load(f)

model = WaveletFlowModule.from_checkpoint(ckpt_path, map_location=device)
model.eval()
model.to(device)

print("Model loaded from checkpoint.")

/Users/gabrieleaccarino/Workspace/wavelet_flow_matching/wavelet_flow_matching/modules/wavelet_fm.py:912: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path

Model loaded from checkpoint.


In [4]:
dataset = get_well_dataset(
    path            = well_base_path,
    physics         = well_dataset_name,
    split           = 'test',
    steps           = -1,
    transform       = ['log_scalars'],   
    augment         = [],
)

In [5]:
N_TRAJ             = len(dataset.metadata.n_steps_per_trajectory)                # number of trajectories in the test set for this 
N_STEPS_INPUT      = 4                                                           # Ti — sliding-window width (must match training)
N_ROLLOUT_STEPS    = dataset.metadata.n_steps_per_trajectory[0] - N_STEPS_INPUT  # autoregressive steps after the seed window
NUM_SAMPLING_STEPS = 20                                                          # ODE steps per prediction
N_ENSEMBLE         = 1                                                           # flow-matching ensemble size

In [6]:
waveletfm_preds = []

with torch.inference_mode():
    for traj_idx in range(N_TRAJ):
        print(f"\n{'━'*60}")
        print(f"  Trajectory {traj_idx}")
        print(f"{'━'*60}")

        item          = dataset[traj_idx]
        x_full, label = get_well_inputs(item, device=device)  # (T, H, W, C), (num_cond,)
        
        seed = x_full[:N_STEPS_INPUT]                         # (N_STEPS_INPUT, H, W, C)

        # Batch-ensemble trick: expand seed across ensemble members
        seed_exp  = seed.unsqueeze(0).expand(N_ENSEMBLE, -1, -1, -1, -1)                      # (N_ENS, T, H, W, C)
        label_exp = label.unsqueeze(0).expand(N_ENSEMBLE, -1) if label is not None else None  # (N_ENS, num_cond) or None
            
        _, preds_phys = model.rollout(
            seed             = seed_exp,        # (N_ENS, N_STEPS_INPUT, H, W, C)
            scalar_conds     = label_exp,       # (N_ENS, num_cond) or None
            rollout_steps    = N_ROLLOUT_STEPS, # N_ROLLOUT_STEPS - N_STEPS_INPUT
            n_steps_input    = N_STEPS_INPUT,   # N_STEPS_INPUT == Ti and must match training config
            n_ensemble       = 1,               # 1 member per batch element
            normalized_input = False,
            verbose          = True,
        )
        # preds: (N_ENS, 1, N_ROLLOUT_STEPS, H, W, C)
        preds_phys = preds_phys.squeeze(1)      # (N_ENS, N_ROLLOUT_STEPS, H, W, C)

        waveletfm_preds.append(preds_phys.cpu())

# (N_TRAJ, N_ENS, N_ROLLOUT_STEPS, H, W, C)  — 6-D, with ensemble dim
waveletfm_preds = torch.stack(waveletfm_preds, dim=0)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call   97/97 (frames 97–97/97)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call   97/97 (frames 97–97/97)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call   97/97 (frames 97–97/97)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 3
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call   97/97 (frames 97–97/97)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 4
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call   97/97 (frames 97–97/97)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 5
━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Save to binary

In [7]:
# save predicted trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results')
save_dir.mkdir(exist_ok=True)

save_path = save_dir / f"{ckpt_base_name}_rollout_nsampling_steps{NUM_SAMPLING_STEPS}_nensemble{N_ENSEMBLE}_traj.pkl"

with open(save_path, "wb") as f:
    pickle.dump(waveletfm_preds, f)